# RDataFrame with collections

In many physics analyses, one event does not contain just one object.

For example, one event may contain:

```text
event 0: particles = [p0, p1]
event 1: particles = [p0]
event 2: particles = [p0, p1, p2, p3]
```

This means that some TTree branches are not simple numbers. They are **collections**.

RDataFrame reads such collections as `ROOT::VecOps::RVec`, often simply called **RVec**.

An RVec behaves a bit like a NumPy array for each event: we can perform vectorized operations, build masks, and select elements.

## Open the collection dataset

This dataset contains one row per event.

Some columns, such as `px`, `py`, and `E`, are collections: each row contains several values.

In [ ]:
import ROOT

treename = "Events"
filename = "data/particles.root"

df = ROOT.RDataFrame(treename, filename)

print("Columns in the dataset:")
for column in df.GetColumnNames():
    print(f"  - {column}: {df.GetColumnType(column)}")

## Inspect a few events

`AsNumpy` converts selected columns to Python/NumPy objects.

This is useful for inspection, but it is not the recommended way to process large datasets.
For real analysis, we will use RDataFrame transformations directly.

In [ ]:
npy = df.AsNumpy(["E", "nPart", "px", "py", "pz"])

for row, energy in enumerate(npy["E"]):
    print(f"\nEvent {row}")
    print(f"  nPart = {npy['nPart'][row]}")
    print(f"  E     = {energy}")
    print(f"  px    = {npy['px'][row]}")
    print(f"  py    = {npy['py'][row]}")
    print(f"  pz    = {npy['pz'][row]}")

## Scalar branch vs collection branch

A scalar branch contains one value per event:

```text
nPart = 3
```

A collection branch contains several values per event:

```text
E = [120, 80, 30]
```

This distinction matters.

When we write:

```cpp
E > 100
```

we do not get one boolean.  
We get a boolean mask for each event:

```text
[120, 80, 30] > 100  →  [true, false, false]
```

## Define transverse momentum for every particle

For each particle we can compute:

```text
pt = sqrt(px*px + py*py)
```

Because `px` and `py` are RVecs, this operation is applied element-by-element.

In [ ]:
df_pt = df.Define("pt", "sqrt(px*px + py*py)")

npy_pt = df_pt.AsNumpy(["pt"])

for row, pt in enumerate(npy_pt["pt"]):
    print(f"Event {row}: pt = {pt}")

## Select elements using a mask

Now we keep only particles with energy above 100:

```cpp
pt[E > 100]
```

Read this as:

> compute `pt`, then keep only the elements for which `E > 100`.

This is one of the most important ideas in collection analysis.

In [ ]:
df_good = df_pt.Define("good_pt", "pt[E > 100]")

npy_good = df_good.AsNumpy(["good_pt"])

for row in range(len(npy_good["good_pt"])):
    print(f"\nEvent {row}")
    print(f"  good_pt = {npy_good['good_pt'][row]}")

## Count selected objects per event

The number of selected particles is obtained with:

```cpp
Sum(E > 100)
```

because `E > 100` is a boolean mask.

For each event, `Sum(mask)` counts how many `true` values are in the mask.

In [ ]:
df_counts = df_good.Define("nGoodParticles", "Sum(E > 100)")

df_counts.Display(["nPart", "nGoodParticles"], nRows=10).Print()

## Filter events using collection information

Now that we have `nGoodParticles`, we can select events.

For example, keep only events with at least one particle with `E > 100`.

In [ ]:
df_events = df_counts.Filter("nGoodParticles >= 1", "At least one particle with E > 100")

print(f"Events before selection: {df.Count().GetValue()}")
print(f"Events after selection: {df_events.Count().GetValue()}")

## Fill histograms from collections

When you pass a collection column to `Histo1D`, RDataFrame fills the histogram with all elements of the collection.

So a histogram of `good_pt` contains the transverse momenta of all selected particles in all selected events.

In [ ]:
canvas = ROOT.TCanvas()

h_good_pt = df_events.Histo1D(
    ("h_good_pt", "Particles with E > 100;p_{T} (GeV);Entries", 40, 0, 200),
    "good_pt",
)

h_good_pt.Draw()
canvas.Draw()

## Leading object example

A common analysis task is to find the leading object, meaning the object with the largest transverse momentum.

For this small tutorial dataset, we can use:

```cpp
Max(pt)
```

after requiring that the event contains at least one particle.

In real analyses, be careful with empty collections before accessing elements or computing leading/subleading quantities.

In [ ]:
df_lead = (
    df_pt
    .Filter("nPart > 0")
    .Define("lead_pt", "Max(pt)")
)

df_lead.Display(["nPart", "pt", "lead_pt"]).Print()

## Exercise: select high-pt particles

Create an analysis that:

1. defines `pt = sqrt(px*px + py*py)`,
2. defines `high_pt = pt[pt > 2.0]`,
3. defines `nHighPt = Sum(pt > 2.0)`,
4. keeps events with at least two high-pt particles,
5. fills a histogram of `high_pt`.

This is the collection-analysis analogue of a typical object selection in particle physics.

In [ ]:
# YOUR SOLUTION

## Common patterns

| Task | RVec expression |
|---|---|
| Build a mask | `E > 100` |
| Select elements | `pt[E > 100]` |
| Count selected elements | `Sum(E > 100)` |
| Require at least one object | `Filter("Sum(E > 100) >= 1")` |
| Compute leading value | `Max(pt)` |
| Fill object-level histogram | `Histo1D(..., "good_pt")` |

The key idea is that RDataFrame lets us work naturally with **event-level data** and **object-level collections** in the same workflow.

This is the basis for many real physics selections, for example:

```cpp
Muon_pt[Muon_pt > 25]
Jet_pt[Jet_btag > 0.6]
Sum(Jet_pt > 25) >= 2
```